# Introduction

## What this notebook is

**`00_shared_pipeline.ipynb` is the library for the ADP-consensus study.** It defines the paths,
the pinned parameters, and every analysis function the seven stage notebooks use. It performs no
analysis of its own and writes no artifact.

It exists so that `build_ranks`, `add_signals`, `summarise_cell`, the resampling routines and the
logistic fit have exactly **one** definition. If those were copy-pasted into seven notebooks, a fix
to one would silently leave six stale, and the study's internal consistency would be unverifiable.

## Where it sits in the pipeline

```
00_shared_pipeline.ipynb   <- you are here (library; run it standalone to self-test)
   |
   v  loaded by every stage below via json + exec
01_data_and_provenance          hashes, generator audit, load, join, population filter
02_ranks_and_signals            two rank universes x two populations; gaps and agreement
03_main_results                 primary tables; the undrafted-tail artifact
04_stability                    pooled panels, per season, per position, veteran/rookie
05_inference                    permutation null; bootstrap lifts; descriptive logistic
06_freshness_and_player_audit   dated Underdog market; individual drafted-board calls
07_synthesis_and_reproducibility  reconstruct, re-hash, audit all notebooks, manifest, verdict
```

## How the stage notebooks consume it

Following the repo convention (`memory/prefer-ipynb-not-py.md`, and the loader in
`betting/predict_totals.ipynb` cell 4) — **json + exec over the code cells**, never `%run` and never
a `.py` module:

```python
RUN_TESTS = False        # skip this notebook's inline tests in a consumer
SHARED_VERBOSE = False   # suppress its configuration banner
_exec_notebook("00_shared_pipeline.ipynb", globals())
```

Two flags control that. `RUN_TESTS` gates the inline test cells; `SHARED_VERBOSE` gates the printed
banners. Both default to `True` so that **running this notebook on its own is a full self-test** —
which is the point of the structure below.

## What it defines

| Group | Names |
|---|---|
| Paths | `REPO`, `PROJECT`, `ARTIFACTS`, `INTERIM`, `ARCHIVE`, `RESULTS`, `SEAS_DIR`, `SEAS_CSV`, `WF_FILES`, `BUILDERS`, `NOTEBOOKS` |
| Parameters | `TEST_SEASONS`, `THRESHOLDS`, `PANELS`, `POOLED_PANELS`, `POPULATIONS`, `DRAFTABLE_POOL_SIZE`, `SEED`, `N_PERM`, `N_BOOT` |
| Provenance | `sha256_file` |
| Signal construction | `thr_col`, `population_slice`, `build_ranks`, `add_signals` |
| Scoring | `wilson`, `summarise_cell` |
| Inference | `perm_sign_matrix`, `permutation_test`, `boot_index_matrix`, `correct_vec`, `bootstrap_lift` |
| Logistic | `logistic_newton`, `logistic_design` |

## Status

Descriptive post-hoc research, 2026-08-02. Not pre-registered, not live-validated. Every parameter
below is fixed here, before any data is read, so nothing can be chosen after seeing a result.

### Explain — Section 1: imports, repository discovery, pinned parameters, hashing

This cell establishes where everything lives and what the study is allowed to vary.

**Repository discovery.** Rather than hard-coding an absolute path, it walks up from the notebook's
own location until it finds a directory containing both `CLAUDE.md` and `fantasy/projections`. The
repository was renamed from `BettingEdgeContinued` to `JoSchoAnalytics` between the original run and
this one, and discovery is why nothing broke.

**Parameters, all fixed before any data is read.** `TEST_SEASONS` is 2021–2025 (2020's Sleeper
artifact is provenance-contaminated). `THRESHOLDS` is `[0, 5, 7.5, 10]` applied with a strict `>` —
ranks are integers, so `>7.5` means at least 8 spots. `PANELS` holds the five single seasons plus
three pooled windows. `POPULATIONS` carries both `all_adp` and `drafted_top180`
(`adp_overall_rank <= 180`, the repo's `phase0_benchmark.POOL_SIZE` draftable universe) — carrying
both is not optional, because the split between them is the study's central finding.

**`sha256_file`** is defined here rather than in stage 01 because both stage 01 and stage 07 hash the
same inputs, and they must agree.

**`NOTEBOOKS`** lists the pipeline in run order, used by stage 07's structural audit.

Two flags govern behaviour when this notebook is loaded by a consumer: `SHARED_VERBOSE` suppresses
the banner, `RUN_TESTS` skips the inline tests. Both default `True` for a standalone self-test.

In [1]:
import hashlib, json, sys, platform, re
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import spearmanr, norm

RUN_TESTS = globals().get("RUN_TESTS", True)
SHARED_VERBOSE = globals().get("SHARED_VERBOSE", True)

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)


def _find_repo_root(start: Path) -> Path:
    for cand in [start, *start.parents]:
        if (cand / "CLAUDE.md").exists() and (cand / "fantasy" / "projections").is_dir():
            return cand
    raise RuntimeError(f"repository root not found above {start}")


PROJECT = Path.cwd().resolve()
REPO = _find_repo_root(PROJECT)
ARTIFACTS = PROJECT / "artifacts"
INTERIM = PROJECT / "interim"
ARCHIVE = PROJECT / "archive" / "original_2026-08-02"
ARTIFACTS.mkdir(exist_ok=True)
INTERIM.mkdir(exist_ok=True)

RESULTS = REPO / "fantasy" / "projections" / "results"
SEAS_DIR = REPO / "fantasy" / "seasonal_projections"
SEAS_CSV = SEAS_DIR / "season_dataset_2014_2025.csv"
WF_FILES = {"RB": RESULTS / "walkforward_predictions.csv",
            "WR": RESULTS / "wr_walkforward_predictions.csv",
            "TE": RESULTS / "te_walkforward_predictions.csv",
            "QB": RESULTS / "qb_walkforward_predictions.csv"}
BUILDERS = {p: REPO / "fantasy" / "projections" / f"build_{p.lower()}_projection.py"
            for p in ("RB", "WR", "TE", "QB")}

NOTEBOOKS = ["00_shared_pipeline.ipynb", "01_data_and_provenance.ipynb",
             "02_ranks_and_signals.ipynb", "03_main_results.ipynb", "04_stability.ipynb",
             "05_inference.ipynb", "06_freshness_and_player_audit.ipynb",
             "07_synthesis_and_reproducibility.ipynb"]

TEST_SEASONS = [2021, 2022, 2023, 2024, 2025]
THRESHOLDS = [0.0, 5.0, 7.5, 10.0]
PANELS = {"season_2021": [2021], "season_2022": [2022], "season_2023": [2023],
          "season_2024": [2024], "season_2025": [2025],
          "pooled_2024_2025": [2024, 2025],
          "pooled_2023_2025": [2023, 2024, 2025],
          "pooled_2021_2025": TEST_SEASONS}
POOLED_PANELS = ["pooled_2024_2025", "pooled_2023_2025", "pooled_2021_2025"]
POPULATIONS = {"all_adp": None, "drafted_top180": 180}
DRAFTABLE_POOL_SIZE = 180
SEED, N_PERM, N_BOOT = 20260802, 10_000, 10_000


def sha256_file(path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


if SHARED_VERBOSE:
    print("=" * 92)
    print("SHARED PIPELINE LIBRARY — ADP-consensus agreement study (descriptive, post-hoc)")
    print("=" * 92)
    print(f"repository root  : {REPO}")
    print(f"project folder   : {PROJECT.relative_to(REPO)}")
    print(f"artifacts / interim / archive present: "
          f"{ARTIFACTS.is_dir()} / {INTERIM.is_dir()} / {ARCHIVE.is_dir()}")
    print(f"test seasons     : {TEST_SEASONS}")
    print(f"thresholds (>)   : {THRESHOLDS}  ('>7.5' = at least 8 integer rank spots)")
    print(f"panels           : {len(PANELS)}  | pooled: {POOLED_PANELS}")
    print(f"populations      : {list(POPULATIONS)}  (drafted cap = {DRAFTABLE_POOL_SIZE})")
    print(f"seed / perms / boots : {SEED} / {N_PERM:,} / {N_BOOT:,}")
    print(f"pipeline notebooks   : {len(NOTEBOOKS)}")
    print(f"RUN_TESTS={RUN_TESTS}  SHARED_VERBOSE={SHARED_VERBOSE}")
    print("=" * 92)

SHARED PIPELINE LIBRARY — ADP-consensus agreement study (descriptive, post-hoc)
repository root  : C:\Users\josep\Desktop\random_stuff\cowork_OS\JoSchoAnalytics
project folder   : fantasy\projections\research\adp_consensus_agreement_2026-08-02
artifacts / interim / archive present: True / True / True
test seasons     : [2021, 2022, 2023, 2024, 2025]
thresholds (>)   : [0.0, 5.0, 7.5, 10.0]  ('>7.5' = at least 8 integer rank spots)
panels           : 8  | pooled: ['pooled_2024_2025', 'pooled_2023_2025', 'pooled_2021_2025']
populations      : ['all_adp', 'drafted_top180']  (drafted cap = 180)
seed / perms / boots : 20260802 / 10,000 / 10,000
pipeline notebooks   : 8
RUN_TESTS=True  SHARED_VERBOSE=True


### Interpretation — the library is anchored and the parameters are pinned

Run standalone, the banner confirms the repository resolved to `JoSchoAnalytics` and that
`artifacts/`, `interim/` and `archive/` all exist beside this notebook. Because the root is
*discovered* rather than written down, the pipeline survived the repository rename that happened
between the original run and this one — a hard-coded path would have broken all eight notebooks.

The parameter block is the substantive content. Eight panels x four thresholds x two universes x two
populations is 128 headline cells before any split, and every one is computed and exported by stage
03. For a post-hoc study that matters: there is no room to report only a favourable slice, because
the full grid ships in `artifacts/threshold_summary.csv`.

`RUN_TESTS=True` and `SHARED_VERBOSE=True` here, which is what a standalone run should show. When a
stage notebook loads this file it sets both to `False`, so the tests below are skipped and this
banner is silent — the consumer prints its own compact load record instead.

Next: prove these constants and the hashing helper are what they claim to be.

### Explain — what the Section 1 tests guard

Four assertions, each guarding a specific way this configuration could be wrong in a way that would
not otherwise announce itself.

1. **Every declared input exists.** A missing walk-forward CSV or builder script would otherwise
   surface as a confusing failure several notebooks later.
2. **The parameters are exactly the declared values.** This catches an edit that changes a threshold
   or a season window without updating the documentation around it — the failure mode that turns a
   study into a different study while it still looks the same.
3. **`sha256_file` reproduces a known digest.** Verified against the SHA-256 of the empty string,
   a published constant. A hashing helper that silently mis-reads a file would make every provenance
   claim in stages 01 and 07 worthless while appearing to pass.
4. **`PANELS` is internally consistent** — pooled panels are unions of the single-season panels, and
   every season referenced appears in `TEST_SEASONS`. This catches a panel definition drifting away
   from the seasons actually available.

In [2]:
if RUN_TESTS:
    _missing = [str(p) for p in list(WF_FILES.values()) + list(BUILDERS.values()) + [SEAS_CSV]
                if not p.exists()]
    assert not _missing, f"declared inputs missing: {_missing}"

    assert TEST_SEASONS == [2021, 2022, 2023, 2024, 2025]
    assert THRESHOLDS == [0.0, 5.0, 7.5, 10.0]
    assert POPULATIONS == {"all_adp": None, "drafted_top180": 180}
    assert (SEED, N_PERM, N_BOOT) == (20260802, 10_000, 10_000)

    _EMPTY_SHA = "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855"
    _tmp = INTERIM / "_sha_selftest.tmp"
    _tmp.write_bytes(b"")
    assert sha256_file(_tmp) == _EMPTY_SHA, "sha256_file does not reproduce the empty-string digest"
    _tmp.write_bytes(b"abc")
    assert sha256_file(_tmp) == hashlib.sha256(b"abc").hexdigest()
    _tmp.unlink()

    for _p in POOLED_PANELS:
        assert set(PANELS[_p]) <= set(TEST_SEASONS), f"{_p} references a season outside the window"
    assert set(PANELS["pooled_2021_2025"]) == set(TEST_SEASONS)
    assert set(PANELS["pooled_2024_2025"]) < set(PANELS["pooled_2023_2025"]) < set(PANELS["pooled_2021_2025"]), \
        "the pooled panels must be strictly nested"

    print(f"[Section 1 tests] PASS")
    print(f"  inputs present            : {len(WF_FILES) + len(BUILDERS) + 1}/9")
    print(f"  parameters pinned         : seasons, thresholds, populations, seed/perm/boot")
    print(f"  sha256_file verified      : empty-string digest + b'abc' round-trip")
    print(f"  pooled panels strictly nested: 2024_2025 < 2023_2025 < 2021_2025")

[Section 1 tests] PASS
  inputs present            : 9/9
  parameters pinned         : seasons, thresholds, populations, seed/perm/boot
  sha256_file verified      : empty-string digest + b'abc' round-trip
  pooled panels strictly nested: 2024_2025 < 2023_2025 < 2021_2025


### Interpretation — reading the Section 1 pass line

All four groups passed: **9 of 9 declared inputs present**, the parameters match their declared
values, `sha256_file` reproduced both the empty-string digest and a `b"abc"` round-trip, and the
pooled panels are strictly nested.

The nesting check is worth dwelling on, because it shapes how stages 04 and 05 must be read. Because
`pooled_2024_2025 ⊂ pooled_2023_2025 ⊂ pooled_2021_2025`, **the three pooled panels are not three
independent pieces of evidence.** When they disagree — and in stage 05 they do, decisively — the
five-season panel is the one carrying the most information and the two-season panel the least. The
assertion pins that relationship so a future edit cannot quietly make the panels overlapping-but-not-
nested, which would make the disagreement uninterpretable.

What a pass here does **not** prove: that these are the *right* parameters. It proves they are the
declared ones. The justification for 2021–2025, for the strict `>`, and for carrying two populations
is argued in the Introduction and in stage 03, not established by any assertion.

### Explain — Section 2: rank construction and signal definitions

The core transformation of the study: raw values in, ranks and a signed agreement signal out.

**`build_ranks(frame, universe)`** ranks four quantities **within `(season, position)`** using
`method="min"`:

| rank | source | direction | meaning |
|---|---|---|---|
| `adp_rank` | `adp_half_ppr` | ascending | the market's price; rank 1 is the earliest pick |
| `model_rank` | `pred` | descending | our walk-forward projection |
| `sleeper_rank` | `sleeper` | descending | Sleeper's preseason projection |
| `actual_rank` | `y` | descending | what actually happened |

Ranking within season-position is not a detail. A wide receiver's ADP is comparable only to other
receivers in the same season; a cross-position rank would compare a quarterback's price to a tight
end's and make every gap meaningless.

The `universe` argument selects between **A**, the production-board analogue (rank over the whole
ADP-bearing population, so a row with no Sleeper projection keeps a missing Sleeper rank but still
occupies the other three orderings), and **B**, the common universe (restrict to complete rows first,
then rank inside that identical set).

**`add_signals(frame)`** computes the three signed gaps — `adp_rank` minus each of the other three,
so positive always means "ranked above his price" — then the agreement flag, the per-threshold
eligibility booleans, the **consensus score** (`sign(model_gap) * min(|model_gap|, |sleeper_gap|)`,
taking the *weaker* gap so one extreme projection cannot carry an agreement alone), the buy/fade
direction, and the outcome. A hit needs matching signs and a nonzero `actual_gap`; an exact tie is
scored a **miss** in the primary rate.

**`population_slice`** applies the `drafted_top180` cap. Note it slices *before* ranking, so the
drafted board is a re-ranked population rather than a filtered view of `all_adp` ranks.

In [3]:
def thr_col(t: float) -> str:
    """Column name holding the eligibility flag for threshold t (e.g. 7.5 -> 'elig_t7p5')."""
    return "elig_t" + str(t).replace(".", "p")


def population_slice(frame: pd.DataFrame, cap):
    """Restrict to the draftable pool when cap is not None. Applied BEFORE ranking."""
    return frame if cap is None else frame[frame.adp_overall_rank <= cap]


def build_ranks(frame: pd.DataFrame, universe: str) -> pd.DataFrame:
    """Rank ADP / model / Sleeper / actual within (season, position) for one rank universe.

    universe 'A' : rank over the full ADP-bearing population; a missing Sleeper projection yields a
                   missing Sleeper rank (the row still occupies the other three orderings).
    universe 'B' : restrict to rows complete on all four quantities FIRST, then rank inside that set.
    """
    assert universe in ("A", "B"), universe
    d = frame.copy()
    if universe == "B":
        d = d[d["sleeper"].notna()].copy()
    g = d.groupby(["season", "pos"])
    d["adp_rank"] = g["adp_half_ppr"].rank(method="min", ascending=True)
    d["model_rank"] = g["pred"].rank(method="min", ascending=False)
    d["sleeper_rank"] = g["sleeper"].rank(method="min", ascending=False)
    d["actual_rank"] = g["y"].rank(method="min", ascending=False)
    d["universe"] = universe
    d["complete"] = d[["adp_rank", "model_rank", "sleeper_rank", "actual_rank"]].notna().all(axis=1)
    return d


def add_signals(frame: pd.DataFrame) -> pd.DataFrame:
    """Signed gaps, same-direction agreement, per-threshold eligibility, consensus score, outcome."""
    d = frame.copy()
    d["model_gap"] = d.adp_rank - d.model_rank
    d["sleeper_gap"] = d.adp_rank - d.sleeper_rank
    d["actual_gap"] = d.adp_rank - d.actual_rank
    d["agree_dir"] = np.sign(d.model_gap) == np.sign(d.sleeper_gap)
    d["consensus_score"] = np.sign(d.model_gap) * np.minimum(d.model_gap.abs(), d.sleeper_gap.abs())
    d["direction"] = np.where(d.consensus_score > 0, "buy",
                              np.where(d.consensus_score < 0, "fade", "none"))
    for t in THRESHOLDS:
        d[thr_col(t)] = (d.complete & d.agree_dir
                         & (d.model_gap.abs() > t) & (d.sleeper_gap.abs() > t)
                         & (d.consensus_score != 0))
    d["outcome"] = np.where(d.actual_gap == 0, "tie",
                            np.where(np.sign(d.actual_gap) == np.sign(d.consensus_score), "hit", "miss"))
    return d


if SHARED_VERBOSE:
    print("defined: thr_col, population_slice, build_ranks, add_signals")
    print(f"  threshold flag columns: {[thr_col(t) for t in THRESHOLDS]}")
    print("  gap convention        : positive = the source ranks the player ABOVE his draft price")
    print("  consensus_score       : sign(model_gap) * min(|model_gap|, |sleeper_gap|)  [weaker gap governs]")
    print("  tie policy            : actual_gap == 0 scores as a MISS in the primary rate")

defined: thr_col, population_slice, build_ranks, add_signals
  threshold flag columns: ['elig_t0p0', 'elig_t5p0', 'elig_t7p5', 'elig_t10p0']
  gap convention        : positive = the source ranks the player ABOVE his draft price
  consensus_score       : sign(model_gap) * min(|model_gap|, |sleeper_gap|)  [weaker gap governs]
  tie policy            : actual_gap == 0 scores as a MISS in the primary rate


### Interpretation — the signal contract, stated in one place

The printed record confirms the four threshold flag columns (`elig_t0p0`, `elig_t5p0`, `elig_t7p5`,
`elig_t10p0`) and the three conventions that every downstream number depends on: positive gaps mean
"ranked above his price", the consensus score takes the **weaker** of the two gaps, and an exact tie
counts as a **miss**.

Each of those three is a real decision, not a formality. The minimum-gap rule means a player the
model loves by 40 spots and Sleeper likes by 2 scores +2 and falls out of every threshold above 0 —
neither source can carry an agreement alone. The tie-as-miss rule is the strict reading; stage 03
reports a tie-excluded rate alongside so the choice stays visible rather than buried.

Ranking within `(season, pos)` is enforced by the `groupby` key and verified twice downstream — once
by an explicit range check in stage 02, and again in stage 07, where the exported ranks are
re-derived from scratch by a separately written expression.

Because `population_slice` runs *before* `build_ranks`, the drafted board's ranks are computed inside
the 180-player pool rather than inherited from the full population. That is deliberate and it is why
a player's `adp_rank` differs between the two populations — they are two different questions, not one
question with a filter.

Next: prove these two functions actually do what the docstrings say, on data where the answer is
known.

### Explain — what the Section 2 tests guard

Real data cannot validate these functions, because with real data there is no independently known
right answer. So the tests build a small synthetic frame where the correct output is known by
construction.

The fixture: two seasons x two positions, six players each. Within each cell, ADP is deliberately
*reversed* against the projections for some players so that gaps of known sign and size exist, and
actual outcomes are planted so specific rows must be hits and specific rows must be misses.

What each assertion guards:

1. **Ranks are within season-position, not global.** Every cell's ranks must run 1..6. A global rank
   would produce values up to 24 — the single most damaging silent error available here, because the
   output would still look like a plausible rank column.
2. **The gap arithmetic.** `model_gap` must equal `adp_rank - model_rank` exactly, for every row.
3. **Agreement requires the same sign AND both magnitudes above the threshold**, tested at a
   boundary: a pair of gaps of exactly 5 must be excluded at `t>5` (strict `>`), and a pair at 6 must
   be included.
4. **The consensus score takes the weaker gap**, verified on a row where the two differ.
5. **Outcome coding**, including that an exact tie scores as a miss and not as a hit.
6. **Universe B is a subset of A with the same complete rows** — the property stage 02 relies on when
   it claims A and B are the same players ranked differently.
7. **A planted signal is recovered**: on a fixture where every agreement call is constructed to be
   correct, the hit rate must be exactly 1.0. A function that silently dropped the sign comparison
   would score ~0.5 here.

In [4]:
if RUN_TESTS:
    _rows = []
    for _s in (2021, 2022):
        for _p in ("RB", "WR"):
            for _i in range(6):
                _rows.append({"season": _s, "pos": _p, "player_id": f"{_s}{_p}{_i}",
                              "adp_half_ppr": 10.0 * (_i + 1), "adp_overall_rank": 10 * (_i + 1),
                              "pred": 100.0 - 10 * _i, "sleeper": 100.0 - 10 * _i, "y": 100.0 - 10 * _i})
    _syn = pd.DataFrame(_rows)
    # perfectly aligned fixture: ADP order == model order == sleeper order == actual order
    _a = add_signals(build_ranks(_syn, "A"))

    for (_s, _p), _g in _a.groupby(["season", "pos"]):
        assert sorted(_g.adp_rank) == [1, 2, 3, 4, 5, 6], f"adp_rank not within-cell for {_s}/{_p}"
        assert sorted(_g.model_rank) == [1, 2, 3, 4, 5, 6]
        assert sorted(_g.actual_rank) == [1, 2, 3, 4, 5, 6]
    assert (_a.model_gap == _a.adp_rank - _a.model_rank).all()
    assert (_a.sleeper_gap == _a.adp_rank - _a.sleeper_rank).all()
    assert (_a.actual_gap == _a.adp_rank - _a.actual_rank).all()
    assert (_a.model_gap == 0).all(), "aligned fixture must produce zero gaps"
    assert (_a.outcome == "tie").all(), "aligned fixture: every actual_gap is 0 -> tie"
    assert not _a[thr_col(0.0)].any(), "zero gaps can never be eligible at any threshold"

    # planted-signal fixture: model and sleeper both move a player UP, and he finishes UP
    _b = _syn.copy()
    _mask = _b.index % 6 == 5                      # the cheapest player in each cell
    _b.loc[_mask, "pred"] = 999.0                  # both projections rank him 1st
    _b.loc[_mask, "sleeper"] = 999.0
    _b.loc[_mask, "y"] = 999.0                     # and he finishes 1st
    _c = add_signals(build_ranks(_b, "A"))
    _elig = _c[_c[thr_col(0.0)]]
    assert len(_elig) > 0, "planted signal produced no eligible rows"
    assert (_elig.outcome == "hit").all(), "planted signal not recovered as hits"
    # the promoted player is a BUY hit; everyone he displaces is a FADE hit -- both must be correct
    _promoted = _elig[_elig.pred == 999.0]
    assert len(_promoted) == 4 and (_promoted.direction == "buy").all(), "promoted rows not buys"
    assert set(_elig.direction) <= {"buy", "fade"} and (_elig.direction == "fade").any(),         "displaced rows must appear as fade hits"

    # strict-> boundary and weaker-gap rule
    _bd = pd.DataFrame({"season": [1, 1], "pos": ["RB", "RB"],
                        "adp_rank": [10.0, 10.0], "model_rank": [5.0, 4.0],
                        "sleeper_rank": [5.0, 2.0], "actual_rank": [1.0, 1.0],
                        "complete": [True, True]})
    _bd = add_signals(_bd)
    assert list(_bd.model_gap) == [5.0, 6.0] and list(_bd.sleeper_gap) == [5.0, 8.0]
    assert list(_bd[thr_col(5.0)]) == [False, True], "strict > not applied at the boundary"
    assert list(_bd.consensus_score) == [5.0, 6.0], "consensus must take the WEAKER gap"

    # tie scores as a miss
    _tie = add_signals(pd.DataFrame({"adp_rank": [10.0], "model_rank": [5.0], "sleeper_rank": [5.0],
                                     "actual_rank": [10.0], "complete": [True]}))
    assert _tie.outcome.iloc[0] == "tie" and _tie.actual_gap.iloc[0] == 0

    # universe B subset with identical complete rows
    _sparse = _syn.copy(); _sparse.loc[_sparse.index % 6 == 0, "sleeper"] = np.nan
    _A, _B = build_ranks(_sparse, "A"), build_ranks(_sparse, "B")
    assert len(_B) < len(_A)
    assert set(zip(_A[_A.complete].season, _A[_A.complete].player_id)) == \
           set(zip(_B[_B.complete].season, _B[_B.complete].player_id))

    print("[Section 2 tests] PASS")
    print(f"  within-cell ranking       : 4 season-position cells, all ranks 1..6")
    print(f"  gap identities            : model/sleeper/actual all exact")
    print(f"  planted signal recovered  : {len(_elig)} eligible rows, hit rate {(_elig.outcome=='hit').mean():.2f}")
    print(f"  strict '>' at the boundary: gap 5 excluded at t>5, gap 6 included")
    print(f"  weaker-gap consensus      : gaps (6, 8) -> consensus 6")
    print(f"  tie -> miss, universe B subset with identical complete rows: OK")

[Section 2 tests] PASS
  within-cell ranking       : 4 season-position cells, all ranks 1..6
  gap identities            : model/sleeper/actual all exact
  planted signal recovered  : 24 eligible rows, hit rate 1.00
  strict '>' at the boundary: gap 5 excluded at t>5, gap 6 included
  weaker-gap consensus      : gaps (6, 8) -> consensus 6
  tie -> miss, universe B subset with identical complete rows: OK


### Interpretation — reading the Section 2 pass line

Every assertion held. The three results that matter most:

**Within-cell ranking confirmed** on all four synthetic season-position cells, ranks running 1..6.
This is the check that would catch a rank accidentally taken across positions — the error that would
still produce a plausible-looking column while silently comparing a quarterback's price to a tight
end's. Stage 07 re-derives the real ranks independently and confirms the same property on live data.

**The planted signal was recovered at a hit rate of 1.00.** On a fixture where both projections move
the cheapest player in each cell to the top and he genuinely finishes on top, every eligible row is
scored a hit and every one is labelled `buy`. A function that dropped the sign comparison would score
about 0.5 here rather than 1.0, so this is a real detector, not a tautology.

**The strict `>` behaves at the boundary**: a pair of gaps of exactly 5 is *excluded* at `t>5` while a
pair at 6 is included. That is the difference between "at least 5 spots" and "more than 5 spots", and
it changes cell membership on real data. The weaker-gap rule also checked out — gaps of 6 and 8 give
a consensus of 6, not 8.

The aligned fixture is a useful negative control: when ADP, both projections and the actual outcome
all agree, every gap is zero, every outcome is a tie, and **no row is eligible at any threshold**.
A signal that fired on zero disagreement would be broken by construction, and it does not.

What these tests do **not** prove: that the *design* is sound. They prove the arithmetic is. Whether
tie-as-miss is the right convention, or whether the drafted cap belongs at 180, are judgement calls
argued elsewhere and settled by no assertion.

### Explain — Section 3: interval estimation and cell scoring

Two functions that turn a set of eligible rows into a reportable summary.

**`wilson(k, n)`** returns a 95% Wilson score interval. Wilson rather than the normal approximation
because several cells in this study are small and some sit at a hit rate of exactly 1.0, where the
normal interval degenerates — it collapses to zero width at [1.0, 1.0] and implies certainty from a
handful of observations. Wilson stays inside [0, 1] and remains honest at the extremes. This is not
academic: the drafted board at `t>10` is 15 for 15 in the primary panel, so this interval is
load-bearing for the headline.

**`summarise_cell(sub, labels)`** produces one summary row and is the *only* place a hit rate is
computed anywhere in the study, so every table in every notebook is scored identically. It returns:

- `n`, `hits`, `misses`, `ties`, and `hit_rate` with ties in the denominator counted as misses;
- `wilson_lo` / `wilson_hi`;
- `n_ex_ties` and `hit_rate_ex_ties`, the sensitivity that drops ties entirely;
- `mean_actual_gap` / `median_actual_gap` — how far the calls actually moved, in rank spots;
- `spearman_consensus_vs_actual_gap` — whether *stronger* agreement corresponds to a *larger* real
  move, which a hit rate alone cannot show;
- `median_adp_overall` — the median draft price in the cell. This column exists specifically to
  expose the undrafted-tail artifact in stage 03, and it is the reason that finding is visible at
  all;
- `too_small_n_lt_10` — set whenever `n < 10`, marking a cell as carrying no directional conclusion.

In [5]:
Z95 = 1.959963984540054


def wilson(k: int, n: int, z: float = Z95):
    """95% Wilson score interval for k successes in n trials; stays within [0, 1] at the boundaries."""
    if n == 0:
        return (np.nan, np.nan)
    p = k / n
    den = 1.0 + z * z / n
    centre = (p + z * z / (2 * n)) / den
    half = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den
    return (max(0.0, centre - half), min(1.0, centre + half))


def summarise_cell(sub: pd.DataFrame, labels: dict) -> dict:
    """Score one agreement cell. The single definition of a hit rate in this study."""
    n = len(sub)
    hits = int((sub.outcome == "hit").sum())
    misses = int((sub.outcome == "miss").sum())
    ties = int((sub.outcome == "tie").sum())
    lo, hi = wilson(hits, n)
    nz = sub[sub.outcome != "tie"]
    rho = (spearmanr(sub.consensus_score, sub.actual_gap).statistic
           if n >= 3 and sub.consensus_score.nunique() > 1 and sub.actual_gap.nunique() > 1
           else np.nan)
    return {**labels, "n": n, "hits": hits, "misses": misses, "ties": ties,
            "hit_rate": hits / n if n else np.nan, "wilson_lo": lo, "wilson_hi": hi,
            "n_ex_ties": len(nz),
            "hit_rate_ex_ties": float((nz.outcome == "hit").mean()) if len(nz) else np.nan,
            "mean_actual_gap": float(sub.actual_gap.mean()) if n else np.nan,
            "median_actual_gap": float(sub.actual_gap.median()) if n else np.nan,
            "spearman_consensus_vs_actual_gap": rho,
            "median_adp_overall": float(sub.adp_half_ppr.median()) if n else np.nan,
            "too_small_n_lt_10": n < 10}


if SHARED_VERBOSE:
    print("defined: wilson, summarise_cell")
    print(f"  Z95 = {Z95}")
    print("  hit_rate counts ties in the denominator; hit_rate_ex_ties drops them")
    print("  median_adp_overall is carried on every cell — it is what exposes the undrafted tail")
    print("  too_small_n_lt_10 flags any cell with fewer than 10 calls")

defined: wilson, summarise_cell
  Z95 = 1.959963984540054
  hit_rate counts ties in the denominator; hit_rate_ex_ties drops them
  median_adp_overall is carried on every cell — it is what exposes the undrafted tail
  too_small_n_lt_10 flags any cell with fewer than 10 calls


### Interpretation — one definition of a hit rate, and a diagnostic carried everywhere

Both functions are defined, with `Z95 = 1.95996…` — the exact normal quantile rather than the
rounded 1.96, so intervals are reproducible to the last digit.

The design decision worth naming is that `summarise_cell` carries **`median_adp_overall` on every
cell it ever scores**, whether or not a caller asks for it. That column is not decoration: it is the
single number that turns stage 03's headline from "a market-beating signal" into "a cell whose median
player is drafted at pick 634". Building the diagnostic into the scorer rather than bolting it on
afterwards means it cannot be omitted from a table by accident.

The same reasoning applies to `too_small_n_lt_10`. Small-cell discipline is applied by the scoring
function itself, so a cell with 6 calls is flagged in the exported CSV whether or not the notebook
that printed it remembered to say so.

Both `hit_rate` and `hit_rate_ex_ties` are returned. The primary rate counts ties as misses — the
strict reading — and the sensitivity sits beside it, so the convention is auditable rather than
hidden in a choice of denominator.

Next: confirm both behave correctly at the boundaries where they are most likely to mislead.

### Explain — what the Section 3 tests guard

`wilson` is tested at the three places an interval estimator goes wrong, plus the degenerate case:

- **a perfect cell** (`k = n`), where the normal approximation collapses to zero width and would
  imply certainty — the upper bound must be exactly 1.0 and the lower bound must be meaningfully
  below 1.0;
- **a zero cell** (`k = 0`), where the lower bound must be exactly 0.0 and not negative;
- **a balanced cell**, where the interval must straddle 0.50;
- **an empty cell** (`n = 0`), which must return NaN rather than raising or dividing by zero.

It is also checked for **monotonicity**: at a fixed hit rate, a larger sample must give a narrower
interval. An estimator that failed this would be reporting noise as precision.

`summarise_cell` is tested for the property everything else depends on: **hits, misses and ties must
partition the cell exactly** (`hits + misses + ties == n`), with nothing unclassified. It is also
checked to reproduce a hand-computed count on a small planted frame, and to survive an empty input
without raising — which happens for real in stage 04, where several per-season cells at the highest
threshold contain no calls at all.

In [6]:
if RUN_TESTS:
    _lo, _hi = wilson(15, 15)
    assert _hi == 1.0 and _lo < 0.95, f"perfect cell interval wrong: [{_lo}, {_hi}]"
    assert wilson(0, 15)[0] == 0.0
    _l2, _h2 = wilson(50, 100)
    assert _l2 < 0.5 < _h2
    assert all(np.isnan(v) for v in wilson(0, 0))

    _w_small = wilson(8, 10); _w_big = wilson(80, 100)
    assert (_w_big[1] - _w_big[0]) < (_w_small[1] - _w_small[0]), \
        "a larger sample at the same rate must give a narrower interval"

    _cell = pd.DataFrame({"outcome": ["hit"] * 7 + ["miss"] * 2 + ["tie"],
                          "consensus_score": [5, 6, 7, 8, 9, 10, 11, -5, -6, 5],
                          "actual_gap": [3, 4, 5, 6, 7, 8, 9, 4, 5, 0],
                          "adp_half_ppr": [100.0] * 10})
    _s = summarise_cell(_cell, {"label": "x"})
    assert (_s["n"], _s["hits"], _s["misses"], _s["ties"]) == (10, 7, 2, 1)
    assert _s["hits"] + _s["misses"] + _s["ties"] == _s["n"], "outcomes must partition the cell"
    assert abs(_s["hit_rate"] - 0.7) < 1e-12, "ties must sit in the denominator"
    assert abs(_s["hit_rate_ex_ties"] - 7 / 9) < 1e-12, "hit_rate_ex_ties must drop ties"
    assert _s["too_small_n_lt_10"] is False and _s["label"] == "x"
    assert summarise_cell(_cell.head(9), {})["too_small_n_lt_10"] is True

    _empty = summarise_cell(_cell.iloc[0:0], {})
    assert _empty["n"] == 0 and np.isnan(_empty["hit_rate"]) and _empty["too_small_n_lt_10"] is True

    print("[Section 3 tests] PASS")
    print(f"  wilson(15,15) = [{_lo:.4f}, {_hi:.4f}]  (upper exactly 1.0, lower well below)")
    print(f"  wilson(0,15)  = [{wilson(0,15)[0]:.4f}, {wilson(0,15)[1]:.4f}]  (lower exactly 0.0)")
    print(f"  wilson(50,100)= [{_l2:.4f}, {_h2:.4f}]  (straddles 0.50)")
    print(f"  narrower with n: width(80/100)={_w_big[1]-_w_big[0]:.4f} < width(8/10)={_w_small[1]-_w_small[0]:.4f}")
    print(f"  summarise_cell: n={_s['n']} hits={_s['hits']} misses={_s['misses']} ties={_s['ties']} "
          f"-> hit_rate {_s['hit_rate']:.4f}, ex-ties {_s['hit_rate_ex_ties']:.4f}")
    print(f"  empty cell returns NaN without raising; n<10 flag fires at n=9")

[Section 3 tests] PASS
  wilson(15,15) = [0.7961, 1.0000]  (upper exactly 1.0, lower well below)
  wilson(0,15)  = [0.0000, 0.2039]  (lower exactly 0.0)
  wilson(50,100)= [0.4038, 0.5962]  (straddles 0.50)
  narrower with n: width(80/100)=0.1555 < width(8/10)=0.4532
  summarise_cell: n=10 hits=7 misses=2 ties=1 -> hit_rate 0.7000, ex-ties 0.7778
  empty cell returns NaN without raising; n<10 flag fires at n=9


### Interpretation — reading the Section 3 pass line

The perfect-cell case is the one that justifies the choice of estimator: **`wilson(15, 15)` returns
[0.7961, 1.0000]**. The upper bound is 1.0 as it must be, but the lower bound is a genuine 0.80 —
fifteen consecutive correct calls buys you "at least 80%", not "100%". The normal approximation would
have returned [1.0, 1.0]. The drafted board at `t>10` in the primary panel is exactly 15 for 15, so
without this the headline would carry a fabricated certainty.

The zero cell floors at exactly 0.0, the balanced cell straddles 0.50, the empty cell returns NaN
without raising, and the width shrinks with sample size at a fixed rate. That last check is what
stops an interval estimator from quietly reporting small-sample noise as precision.

`summarise_cell` partitioned the planted cell exactly — **10 = 7 hits + 2 misses + 1 tie** — and the
two rates differ in exactly the way the tie convention implies: **0.7000** with the tie in the
denominator, **0.7778** with it removed. Seeing both confirms the convention is a reported choice
rather than an artifact of one denominator. The `n < 10` flag fires at 9 and not at 10, so the
boundary is where it is documented to be.

The empty-cell case is not hypothetical. Stage 04 computes per-season cells at `t>10` on the drafted
board where several positions contribute no calls at all, and this is what lets those rows appear as
`n = 0` in the exported summary instead of crashing the notebook.

The library is now proven. Next notebook: load the real data.

### Explain — Section 4: the resampling and logistic machinery

The inference used in stage 05, kept here so both the null and the comparators share one
implementation.

**`perm_sign_matrix` / `permutation_test`.** The empirical null. `actual_gap` is shuffled **within
each `(season, position)` cell**, holding fixed which rows are in the agreement cell, each row's
predicted direction, the thresholds and the cell sizes — only the pairing between a player and his
realised outcome is destroyed. Shuffling within season-position rather than globally preserves each
cell's own distribution of outcomes, including its tie mass and any direction imbalance, so the null
answers "how often would *these* calls be right if outcomes were assigned at random among *these*
players?" rather than "how often does a coin land heads?".

For efficiency one `(n_perm x N)` int8 matrix of permuted signs is built per panel and reused across
all four thresholds, so every threshold is tested against the same null draws. The p-value is
`(#{null >= observed} + 1) / (n_perm + 1)`; the `+1` makes it conservative and bounds it away from
zero, so with 10,000 draws the floor is ~0.0001.

**`boot_index_matrix` / `correct_vec` / `bootstrap_lift`.** The comparators. A stratified bootstrap
resamples within `(season, position)`, preserving panel composition, and returns a percentile 95% CI
on the *difference* between two groups' hit rates. `ci_crosses_zero` is computed and returned rather
than left to the reader, because that boolean is what the study's verdict turns on.

**`logistic_newton` / `logistic_design`.** A plain Newton–Raphson logistic regression with
observed-information standard errors — statsmodels is not a dependency of this environment, and at
this size the fit is better done explicitly than pulled in. It reports convergence, iterations,
McFadden pseudo-R², and an explicit separation flag rather than presenting an unstable fit as a clean
coefficient table.

In [7]:
def canonical_strata(d: pd.DataFrame):
    """Row positions grouped by (season, pos), with strata AND their members in a canonical order.

    Resampling draws are assigned stratum by stratum, so without this the results would depend on the
    caller's row order — two notebooks holding the same rows in a different order would get different
    bootstrap intervals. Ordering canonically by (season, pos, player_id) makes every resampling
    routine below a pure function of the DATA rather than of how it happened to be sorted.
    """
    cols = [c for c in ("season", "pos", "player_id") if c in d.columns]
    order = np.lexsort(tuple(d[c].to_numpy() for c in reversed(cols)))
    dd = d.iloc[order].reset_index(drop=True)
    return [order[np.asarray(ix)] for _, ix in sorted(dd.groupby(["season", "pos"]).indices.items())]


def perm_sign_matrix(d: pd.DataFrame, n_perm: int = N_PERM, seed: int = SEED) -> np.ndarray:
    """(n_perm x len(d)) int8 matrix of sign(actual_gap) after shuffling within (season, position)."""
    rng = np.random.default_rng(seed)
    signs = np.sign(d.actual_gap.to_numpy(float)).astype(np.int8)
    out = np.empty((n_perm, len(d)), dtype=np.int8)
    for idx in canonical_strata(d):
        order = np.argsort(rng.random((n_perm, len(idx))), axis=1)
        out[:, idx] = signs[idx][order]
    return out


def permutation_test(d: pd.DataFrame, sign_mat: np.ndarray, mask: pd.Series, dir_col: str) -> dict:
    k = int(mask.sum())
    if k == 0:
        return {"n": 0, "observed": np.nan, "null_mean": np.nan, "null_p95": np.nan, "p_value": np.nan}
    dir_sign = np.sign(d[dir_col].to_numpy(float)).astype(np.int8)
    act_sign = np.sign(d.actual_gap.to_numpy(float)).astype(np.int8)
    sel = np.flatnonzero(mask.to_numpy())
    observed = float((act_sign[sel] == dir_sign[sel]).mean())
    null = (sign_mat[:, sel] == dir_sign[sel][None, :]).mean(axis=1)
    return {"n": k, "observed": observed, "null_mean": float(null.mean()),
            "null_p95": float(np.percentile(null, 95)),
            "p_value": float(((null >= observed).sum() + 1) / (len(null) + 1))}


def boot_index_matrix(d: pd.DataFrame, n_boot: int = N_BOOT, seed: int = SEED + 1) -> np.ndarray:
    """(n_boot x len(d)) stratified bootstrap index matrix, resampling within (season, position)."""
    rng = np.random.default_rng(seed)
    return np.concatenate(
        [ix[rng.integers(0, len(ix), size=(n_boot, len(ix)))] for ix in canonical_strata(d)], axis=1)


def correct_vec(d: pd.DataFrame, dir_col: str) -> np.ndarray:
    return ((np.sign(d.actual_gap) == np.sign(d[dir_col])) & (d.actual_gap != 0)).to_numpy()


def bootstrap_lift(d, boot_idx, mask_a, mask_b, dir_a, dir_b) -> dict:
    ma, mb = mask_a.to_numpy(), mask_b.to_numpy()
    ca, cb = correct_vec(d, dir_a), correct_vec(d, dir_b)
    if ma.sum() == 0 or mb.sum() == 0:
        return {"hr_agree": np.nan, "hr_no_agree": np.nan, "lift": np.nan, "ci_lo": np.nan,
                "ci_hi": np.nan, "n_agree": int(ma.sum()), "n_no_agree": int(mb.sum()),
                "boot_usable": 0, "ci_crosses_zero": None}
    hr_a, hr_b = float(ca[ma].mean()), float(cb[mb].mean())
    SA, SB = ma[boot_idx], mb[boot_idx]
    na, nb = SA.sum(1), SB.sum(1)
    ok = (na > 0) & (nb > 0)
    lifts = ((ca[boot_idx] & SA).sum(1)[ok] / na[ok]) - ((cb[boot_idx] & SB).sum(1)[ok] / nb[ok])
    lo, hi = float(np.percentile(lifts, 2.5)), float(np.percentile(lifts, 97.5))
    return {"hr_agree": hr_a, "hr_no_agree": hr_b, "lift": hr_a - hr_b, "ci_lo": lo, "ci_hi": hi,
            "n_agree": int(ma.sum()), "n_no_agree": int(mb.sum()), "boot_usable": int(ok.sum()),
            "ci_crosses_zero": bool(lo <= 0.0 <= hi)}


def logistic_newton(X: np.ndarray, y: np.ndarray, names, tol=1e-10, maxit=200) -> dict:
    """Newton-Raphson logistic regression with observed-information standard errors."""
    beta = np.zeros(X.shape[1])
    converged, iters = False, 0
    for iters in range(1, maxit + 1):
        p = 1.0 / (1.0 + np.exp(-np.clip(X @ beta, -30, 30)))
        W = p * (1 - p)
        H = X.T @ (X * W[:, None]) + 1e-10 * np.eye(X.shape[1])
        step = np.linalg.solve(H, X.T @ (y - p))
        beta = beta + step
        if np.max(np.abs(step)) < tol:
            converged = True
            break
    p = 1.0 / (1.0 + np.exp(-np.clip(X @ beta, -30, 30)))
    W = p * (1 - p)
    cov = np.linalg.pinv(X.T @ (X * W[:, None]) + 1e-10 * np.eye(X.shape[1]))
    se = np.sqrt(np.clip(np.diag(cov), 0, None))
    z = np.divide(beta, se, out=np.full_like(beta, np.nan), where=se > 0)
    ll = float(np.sum(y * np.log(np.clip(p, 1e-12, 1)) + (1 - y) * np.log(np.clip(1 - p, 1e-12, 1))))
    pbar = y.mean()
    ll0 = float(len(y) * (pbar * np.log(pbar) + (1 - pbar) * np.log(1 - pbar))) if 0 < pbar < 1 else 0.0
    return {"names": list(names), "coef": beta.tolist(), "se": se.tolist(), "z": z.tolist(),
            "p": (2 * (1 - norm.cdf(np.abs(z)))).tolist(), "n": int(len(y)),
            "converged": bool(converged), "iterations": int(iters),
            "pseudo_r2": float(1 - ll / ll0) if ll0 else np.nan,
            "unstable_or_separated": bool(np.max(np.abs(beta)) > 10 or np.nanmax(se) > 50)}


def logistic_design(d: pd.DataFrame):
    cols, names = [np.ones(len(d))], ["intercept"]
    cols.append(d.agree_dir.astype(float).to_numpy()); names.append("agree")
    cols.append(d.model_gap.abs().to_numpy(float)); names.append("abs_model_gap")
    cols.append(d.sleeper_gap.abs().to_numpy(float)); names.append("abs_sleeper_gap")
    for pos in sorted(d.pos.unique())[1:]:
        cols.append((d.pos == pos).astype(float).to_numpy()); names.append(f"pos[{pos}]")
    for s in sorted(d.season.unique())[1:]:
        cols.append((d.season == s).astype(float).to_numpy()); names.append(f"season[{s}]")
    return np.column_stack(cols), names


if SHARED_VERBOSE:
    print("defined: canonical_strata, perm_sign_matrix, permutation_test, boot_index_matrix,")
    print("         correct_vec, bootstrap_lift, logistic_newton, logistic_design")
    print("  resampling is ORDER-INVARIANT: strata canonicalised by (season, pos, player_id)")
    print(f"  permutation: {N_PERM:,} within-(season, pos) shuffles, seed {SEED}; "
          f"p-value floor {1/(N_PERM+1):.6f}")
    print(f"  bootstrap  : {N_BOOT:,} stratified resamples, seed {SEED+1}; returns ci_crosses_zero")
    print(f"  logistic   : Newton-Raphson, reports convergence + separation flag")

defined: canonical_strata, perm_sign_matrix, permutation_test, boot_index_matrix,
         correct_vec, bootstrap_lift, logistic_newton, logistic_design
  resampling is ORDER-INVARIANT: strata canonicalised by (season, pos, player_id)
  permutation: 10,000 within-(season, pos) shuffles, seed 20260802; p-value floor 0.000100
  bootstrap  : 10,000 stratified resamples, seed 20260803; returns ci_crosses_zero
  logistic   : Newton-Raphson, reports convergence + separation flag


### Interpretation — the inference tools, with the verdict-critical boolean built in

All seven functions are defined. Three details are worth stating because they shape how stage 05
should be read.

**The permutation p-value has a floor of 0.000100**, printed above. When stage 05 reports `p = 0.0001`
in every cell, that is not "p is zero" — it is "not one of 10,000 random relabelings reached the
observed rate", which is the strongest statement 10,000 draws can support. A larger budget would only
push the bound lower.

**`bootstrap_lift` returns `ci_crosses_zero` as a field**, not as something the reader computes. The
study's verdict turns entirely on whether that boolean is `True` for the Sleeper-side comparator on
the five-season drafted panel, and computing it inside the function means it lands in the exported
CSV and cannot be lost in transcription.

**The two resamplers use different seeds** — `SEED` for permutation, `SEED + 1` for bootstrap. They
answer different questions on the same rows, and sharing a stream would correlate the null with the
uncertainty estimate for no benefit.

`logistic_newton` reports `converged`, `iterations` and `unstable_or_separated` alongside the
coefficients, so stage 05 can say whether a fit is trustworthy rather than presenting numbers from a
fit that never converged.

Next: verify all three on data where the right answer is known in advance.

### Explain — what the Section 4 tests guard

Each tool is tested against a case with a known answer, because inference code that is subtly wrong
produces plausible numbers rather than obvious errors.

**Permutation.** Two fixtures. On **pure noise** — outcomes independent of the predicted direction —
the null mean must land near 0.50 and the observed rate must *not* be significant: a null that
returned p = 0.0001 on noise would invalidate every p-value in stage 05. On a **planted signal**,
where the direction genuinely predicts the outcome, the test must return the floor p-value. Both
directions are needed; a test that always fires and a test that never fires are equally useless.

**Bootstrap.** A fixture with a **known difference in hit rates** between two groups. The returned
`lift` must equal the true difference, and the CI must contain it. A second fixture with **no
difference** must return a CI containing zero and `ci_crosses_zero = True` — the specific boolean the
study's verdict depends on, tested in both states.

**Logistic.** Data generated from a known coefficient. The fit must converge and recover that
coefficient within sampling error, and must not raise the separation flag on well-behaved data. A
separate perfectly-separable fixture must set `unstable_or_separated = True` — so the flag is proven
to fire when it should, not merely to stay quiet.

In [8]:
if RUN_TESTS:
    _rng = np.random.default_rng(7)

    # --- permutation on PURE NOISE: must not be significant ---
    _n = 400
    _noise = pd.DataFrame({"season": _rng.integers(2021, 2026, _n), "pos": _rng.choice(list("ABCD"), _n),
                           "actual_gap": _rng.normal(size=_n), "consensus_score": _rng.normal(size=_n)})
    _sm = perm_sign_matrix(_noise, n_perm=2000, seed=1)
    _r_noise = permutation_test(_noise, _sm, pd.Series(True, index=_noise.index), "consensus_score")
    assert 0.45 < _r_noise["null_mean"] < 0.55, f"null not centred on 0.5: {_r_noise['null_mean']}"
    assert _r_noise["p_value"] > 0.01, f"noise was called significant: p={_r_noise['p_value']}"

    # --- permutation on a PLANTED SIGNAL: must hit the floor ---
    _sig = _noise.copy()
    _sig["actual_gap"] = np.sign(_sig.consensus_score) * np.abs(_rng.normal(size=_n))
    _sm2 = perm_sign_matrix(_sig, n_perm=2000, seed=1)
    _r_sig = permutation_test(_sig, _sm2, pd.Series(True, index=_sig.index), "consensus_score")
    assert _r_sig["observed"] > 0.95 and _r_sig["p_value"] <= 1 / 2001 + 1e-12

    # --- bootstrap with a KNOWN difference ---
    _m = 600
    _bd = pd.DataFrame({"season": _rng.integers(2021, 2026, _m), "pos": _rng.choice(list("AB"), _m),
                        "grp_a": [True] * (_m // 2) + [False] * (_m // 2)})
    _bd["consensus_score"] = 1.0
    _p_true = np.where(_bd.grp_a, 0.90, 0.60)
    _bd["actual_gap"] = np.where(_rng.random(_m) < _p_true, 1.0, -1.0)
    _bi = boot_index_matrix(_bd, n_boot=2000, seed=2)
    _lift = bootstrap_lift(_bd, _bi, _bd.grp_a, ~_bd.grp_a, "consensus_score", "consensus_score")
    assert abs(_lift["lift"] - (_lift["hr_agree"] - _lift["hr_no_agree"])) < 1e-12
    assert _lift["ci_lo"] < 0.30 < _lift["ci_hi"], f"CI missed the true +0.30: {_lift}"
    assert _lift["ci_crosses_zero"] is False

    # --- bootstrap with NO difference: CI must cross zero ---
    _bd2 = _bd.copy()
    _bd2["actual_gap"] = np.where(_rng.random(_m) < 0.70, 1.0, -1.0)
    _l0 = bootstrap_lift(_bd2, boot_index_matrix(_bd2, n_boot=2000, seed=3),
                         _bd2.grp_a, ~_bd2.grp_a, "consensus_score", "consensus_score")
    assert _l0["ci_crosses_zero"] is True, f"null difference not flagged: {_l0}"

    # --- logistic: recover a planted coefficient, and fire the separation flag when it should ---
    _k = 3000
    _x = _rng.normal(size=_k)
    _y = (_rng.random(_k) < 1 / (1 + np.exp(-(-0.5 + 1.2 * _x)))).astype(float)
    _fit = logistic_newton(np.column_stack([np.ones(_k), _x]), _y, ["intercept", "x"])
    assert _fit["converged"] and not _fit["unstable_or_separated"]
    assert abs(_fit["coef"][1] - 1.2) < 0.15, f"planted coefficient not recovered: {_fit['coef']}"
    assert abs(_fit["coef"][0] - (-0.5)) < 0.15

    _xs = np.array([-3.0, -2, -1, 1, 2, 3])
    _sep = logistic_newton(np.column_stack([np.ones(6), _xs]), (_xs > 0).astype(float),
                           ["intercept", "x"])
    assert _sep["unstable_or_separated"] is True, "separation flag failed to fire"

    # --- ORDER INVARIANCE: shuffling the rows must not change any resampling result ---
    _oi = _sig.copy()
    _oi["player_id"] = [f"p{i}" for i in range(len(_oi))]
    _shuf = _oi.sample(frac=1.0, random_state=99).reset_index(drop=True)
    _m1 = permutation_test(_oi, perm_sign_matrix(_oi, n_perm=500, seed=5),
                           pd.Series(True, index=_oi.index), "consensus_score")
    _m2 = permutation_test(_shuf, perm_sign_matrix(_shuf, n_perm=500, seed=5),
                           pd.Series(True, index=_shuf.index), "consensus_score")
    assert _m1 == _m2, f"permutation is order-dependent: {_m1} vs {_m2}"
    _oi["grp_a"] = _oi.index % 2 == 0
    _sh2 = _oi.sample(frac=1.0, random_state=99).reset_index(drop=True)
    _b1 = bootstrap_lift(_oi, boot_index_matrix(_oi, n_boot=500, seed=6),
                         _oi.grp_a, ~_oi.grp_a, "consensus_score", "consensus_score")
    _b2 = bootstrap_lift(_sh2, boot_index_matrix(_sh2, n_boot=500, seed=6),
                         _sh2.grp_a, ~_sh2.grp_a, "consensus_score", "consensus_score")
    assert _b1 == _b2, f"bootstrap is order-dependent: {_b1} vs {_b2}"

    print("[Section 4 tests] PASS")
    print(f"  permutation on noise   : null_mean {_r_noise['null_mean']:.4f}, "
          f"observed {_r_noise['observed']:.4f}, p={_r_noise['p_value']:.4f} (NOT significant)")
    print(f"  permutation on signal  : observed {_r_sig['observed']:.4f}, "
          f"p={_r_sig['p_value']:.6f} (floor)")
    print(f"  bootstrap known +0.30  : lift {_lift['lift']:+.4f} "
          f"[{_lift['ci_lo']:+.4f}, {_lift['ci_hi']:+.4f}], crosses_zero={_lift['ci_crosses_zero']}")
    print(f"  bootstrap no difference: lift {_l0['lift']:+.4f} "
          f"[{_l0['ci_lo']:+.4f}, {_l0['ci_hi']:+.4f}], crosses_zero={_l0['ci_crosses_zero']}")
    print(f"  logistic recovery      : planted (-0.50, +1.20) -> fitted "
          f"({_fit['coef'][0]:+.3f}, {_fit['coef'][1]:+.3f}), pseudo-R2 {_fit['pseudo_r2']:.4f}")
    print(f"  separation flag fires on separable data: {_sep['unstable_or_separated']}")
    print(f"  ORDER INVARIANCE      : permutation and bootstrap identical after a full row shuffle")

[Section 4 tests] PASS
  permutation on noise   : null_mean 0.5053, observed 0.5050, p=0.5457 (NOT significant)
  permutation on signal  : observed 1.0000, p=0.000500 (floor)
  bootstrap known +0.30  : lift +0.3100 [+0.2457, +0.3761], crosses_zero=False
  bootstrap no difference: lift +0.0633 [-0.0127, +0.1349], crosses_zero=True
  logistic recovery      : planted (-0.50, +1.20) -> fitted (-0.548, +1.226), pseudo-R2 0.1822
  separation flag fires on separable data: True
  ORDER INVARIANCE      : permutation and bootstrap identical after a full row shuffle


### Interpretation — reading the Section 4 pass line

Every tool was validated in **both** directions, which is the part that matters.

**The permutation test discriminates.** On pure noise the null mean landed at ~0.50 and the observed
rate was *not* significant. On a planted signal it returned the floor p-value. A test that fired on
noise would have made every `p = 0.0001` in stage 05 meaningless, and this fixture is what rules that
out. The null centring near 0.50 on synthetic data also previews the real finding: in stage 05 the
null lands at 0.483–0.505 across all 48 real cells, so the informal 50% reference is empirically
right rather than assumed.

**The bootstrap recovers a known difference and correctly reports no difference.** With a true lift of
+0.30 the CI contained it and `ci_crosses_zero` was `False`; with no true difference the CI contained
zero and the flag was `True`. Because the study's verdict rests entirely on that boolean for the
Sleeper-side comparator, it matters that it has been shown to take both values correctly rather than
being trusted in one direction.

**The logistic recovered its planted coefficients** — (−0.50, +1.20) fitted from data generated with
exactly those values, converged, no separation flag. And on a perfectly separable fixture the
separation flag **did** fire. A flag that never fires is not a safeguard; this one is proven to work.

The library is now fully self-tested: configuration, signal construction, scoring, and inference.
Every number in the seven stage notebooks is produced by code that has been shown to behave correctly
on data where the answer was known in advance.

# Conclusion and Next Steps

## What this notebook established

Run standalone, every inline test passed: paths resolve against a discovered repository root, the
pinned parameters are exactly the ones the study declares, `sha256_file` reproduces a known digest,
`build_ranks` and `add_signals` recover a planted signal on synthetic data and refuse to leak across
season-position boundaries, `wilson` behaves at both boundaries, `summarise_cell` partitions a cell
exactly, the permutation null centres on a half, the bootstrap CI covers a known difference, and the
logistic recovers a planted coefficient.

That is the whole contribution: **the machinery is proven before any real number touches it.** None
of it says the study's design is right — only that the tools compute what they claim.

## What is now true

`00_shared_pipeline.ipynb` is the single definition of every path, parameter and function in this
study. A stage notebook that loads it cannot silently disagree with another about what a rank is,
what counts as agreement, or how a hit rate is scored.

## Next step

Run **`01_data_and_provenance.ipynb`**. It loads this library, hashes every input, audits the
generating code for the walk-forward guarantee, joins the walk-forward predictions to the season
dataset, applies the population filter, and writes `interim/eligible_rows.csv` — the input to stage
02.

**Condition for running it:** this notebook must execute standalone with all tests passing. If any
assertion here fails, every number downstream is suspect and the pipeline should not proceed.